In [1]:
import pymc3 as pm
import numpy as np
import theano.tensor as tt
import matplotlib.pyplot as plt
from ann_functions import import_data
from keras.models import load_model
from keras.initializers import glorot_uniform
import keras.backend as K
from keras.utils import custom_object_scope



WARN: Could not locate executable g77
WARN: Could not locate executable f77
WARN: Could not locate executable ifort
WARN: Could not locate executable ifl
WARN: Could not locate executable f90
WARN: Could not locate executable DF
WARN: Could not locate executable efl
WARN: Could not locate executable gfortran
WARN: Could not locate executable f95
WARN: Could not locate executable g95
WARN: Could not locate executable efort
WARN: Could not locate executable efc
WARN: Could not locate executable flang
WARN: don't know how to compile Fortran code on platform 'nt'


WARNING (theano.configdefaults): g++ not available, if using conda: `conda install m2w64-toolchain`
WARNING (theano.configdefaults): g++ not detected ! Theano will be unable to execute optimized C-implementations (for both CPU and GPU) and will default to Python implementations. Performance will be severely degraded. To remove this warning, set Theano flags cxx to an empty string.
c:\Users\Giacomo\AppData\Local\Programs\Python\Python310\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.1
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
WARNING (theano.tensor.blas): Using NumPy C-API based implementation for BLAS functions.


In [2]:


file_path_HF = "../DATA/reaction_diffusion_HF.mat"
(reaction_HF_test, U_HF_test) = import_data(file_path_HF)
U_HF_test = U_HF_test[:, -1, 44,44]
# ADD NOISE
noise_stddev = np.mean(U_HF_test) * 0.025       # value?
noise = np.random.normal(0, noise_stddev, len(U_HF_test))
U_HF_test = U_HF_test + noise


mu_0=0.5*np.ones(U_HF_test.shape)


#eps = 0.01
#alpha = 10**3
#sigma = 0.4
#n = 1000
#final_q = np.zeros([n, 2])
#final_q_ham = np.zeros([n, 2])
#ratio = np.zeros(n)
#big_q = np.zeros([n, N+1, 2])
#big_q_ham = np.zeros([n, N+1, 2])


In [3]:
# Definisci la tua funzione di attivazione personalizzata
def custom_activation(x):
    # Implementa la tua logica personalizzata qui
    return x + K.square(K.sin(x))

# Aggiungi la tua funzione di attivazione personalizzata al dizionario custom_objects
custom_objects = {'custom_activation': custom_activation, 'glorot_uniform': glorot_uniform()}

# Carica il modello utilizzando custom_objects
with custom_object_scope(custom_objects):
    keras_model = load_model("best_model.h5")

In [4]:
print(keras_model.predict(mu_0))

ValueError: in user code:

    File "c:\Users\Giacomo\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\engine\training.py", line 2137, in predict_function  *
        return step_function(self, iterator)
    File "c:\Users\Giacomo\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\engine\training.py", line 2123, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "c:\Users\Giacomo\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\engine\training.py", line 2111, in run_step  **
        outputs = model.predict_step(data)
    File "c:\Users\Giacomo\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\engine\training.py", line 2079, in predict_step
        return self(x, training=False)
    File "c:\Users\Giacomo\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\utils\traceback_utils.py", line 70, in error_handler
        raise e.with_traceback(filtered_tb) from None
    File

In [7]:


# Estrai i pesi dalla rete neurale Keras
weights_input_hidden = keras_model.get_layer('nome_layer_input_hidden').get_weights()
weights_hidden_output = keras_model.get_layer('nome_layer_hidden_output').get_weights()
bias_hidden = keras_model.get_layer('nome_layer_hidden').get_weights()
bias_output = keras_model.get_layer('nome_layer_output').get_weights()

# Dati osservati
y_obs = U_HF_test

# Parametri della rete neurale
n_inputs = 1  # Numero di input della rete neurale
n_hidden = 4  # Numero di neuroni nascosti
n_outputs = 1

with pm.Model() as model:
    # Pesi della rete neurale
    pm_weights_input_hidden = pm.Normal('weights_input_hidden', mu=0, sd=1, shape=weights_input_hidden[0].shape, testval=weights_input_hidden[0])
    pm_weights_hidden_output = pm.Normal('weights_hidden_output', mu=0, sd=1, shape=weights_hidden_output[0].shape, testval=weights_hidden_output[0])
    pm_bias_hidden = pm.Normal('bias_hidden', mu=0, sd=1, shape=bias_hidden[0].shape, testval=bias_hidden[0])
    pm_bias_output = pm.Normal('bias_output', mu=0, sd=1, shape=bias_output[0].shape, testval=bias_output[0])

    # Input del passo MCMC come variabile theano condivisa
    input_mcmc_shared = pm.theanoshared(np.zeros(n_inputs), name='input_mcmc_shared', borrow=True)

    # Calcolo della rete neurale utilizzando i pesi della rete Keras
    hidden_activation = tt.tanh(pm.math.dot(input_mcmc_shared, pm_weights_input) + pm_bias_hidden)
    output_activation = pm.math.dot(hidden_activation, pm_weights_hidden_output) + pm_bias_output
    
    # Likelihood
    likelihood = pm.Normal('y', mu=output_activation.flatten(), sd=2, observed=y_obs)

    # Campionamento MCMC
    trace = pm.sample(2000, tune=1000, cores=1)


ValueError: No such layer: nome_layer_input_hidden. Existing layers are: ['input_1', 'dense', 'HF'].

In [ ]:
def custom_activation(x):
    # Implementa la tua logica personalizzata qui
    return x + K.square(K.sin(x))

# Aggiungi la tua funzione di attivazione personalizzata al dizionario custom_objects
custom_objects = {'custom_activation': custom_activation, 'GlorotUniform': glorot_uniform()}

# Carica il modello utilizzando custom_objects
with custom_object_scope(custom_objects):
    model = load_model("best_model.h5")

In [ ]:
mu=mu_0
with pm.Model() as model:
        # Priori
    #slope = pm.Normal('slope', mu=0, sd=10)
    #intercept = pm.Normal('intercept', mu=0, sd=10)
    U_MF=model(mu)
    # Likelihood
    likelihood = pm.Normal('y', mu=U_MF, sd=2, observed=U_HF_test)

    # Creation of Metropolis object
    step = pm.Metropolis()

    # Campionamento MCMC
    trace = pm.sample(2000, tune=1000, cores=1, step=step) # tune := burn-in

# analysis of the summary, resume
pm.summary(trace).round(2)   

# Plo
pm.traceplot(trace)